# Neural Network + FGM Pipeline

This notebook follows the handwritten pipeline and uses the model settings loaded from `MachineLearning\NeuralNetworks\best_params.csv`.

Pipeline steps:

1. `clean.fit(X_train, y_train)` to train the clean model  
2. `clean.predict(X_test)` and evaluate the clean model on clean test data  
3. Save the clean model  
4. Initialize the FGM attack  
5. Generate adversarial train and test samples  
6. Keep adversarial labels aligned with the original ground-truth labels  
7. Build combined clean+adversarial train and test sets  
8. Retrain using ART's `AdversarialTrainer` with the FGM attack  
9. Evaluate both the clean model and the adversarially trained model on:
   - clean test data
   - adversarial test data
   - combined clean+adversarial test data


In [1]:
# If needed, install once:
# !pip install torch scikit-learn adversarial-robustness-toolbox pandas numpy

import warnings
warnings.filterwarnings("ignore")

import os
import ast
import random
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report

import torch
import torch.nn as nn
import torch.optim as optim

from art.estimators.classification import PyTorchClassifier
from art.attacks.evasion import FastGradientMethod
from art.defences.trainer import AdversarialTrainer


In [2]:
BEST_PARAMS_PATH = Path(r"MachineLearning\NeuralNetworks\best_params.csv")

def parse_hidden_layers(value):
    if isinstance(value, (tuple, list)):
        return tuple(int(v) for v in value)

    text = str(value).strip().strip('"').strip("'")
    if text.startswith("(") or text.startswith("["):
        parsed = ast.literal_eval(text)
        if isinstance(parsed, (tuple, list)):
            return tuple(int(v) for v in parsed)

    return tuple(int(part.strip()) for part in text.split(",") if part.strip())

if not BEST_PARAMS_PATH.exists():
    raise ValueError(f"File not found: {BEST_PARAMS_PATH}")

best_params_df = pd.read_csv(BEST_PARAMS_PATH)
BEST_PARAMS = best_params_df.iloc[0].to_dict()
BEST_PARAMS["hidden_layer_sizes"] = parse_hidden_layers(BEST_PARAMS["hidden_layer_sizes"])
BEST_PARAMS["alpha"] = float(BEST_PARAMS["alpha"])
BEST_PARAMS["learning_rate_init"] = float(BEST_PARAMS["learning_rate_init"])
BEST_PARAMS["batch_size"] = int(BEST_PARAMS["batch_size"])
BEST_PARAMS["max_iter"] = int(BEST_PARAMS["max_iter"])
BEST_PARAMS["early_stopping"] = bool(BEST_PARAMS["early_stopping"])
BEST_PARAMS["n_iter_no_change"] = int(BEST_PARAMS["n_iter_no_change"])
BEST_PARAMS["random_state"] = int(BEST_PARAMS["random_state"])

SEED = BEST_PARAMS["random_state"]
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

DEFAULT_DATA_PATH = Path(r"CSVs\newDataset.csv")
RUNS_DIR = Path(r"StandardizedRuns")
RUN_GLOB = "NeuralNet_train_*.csv"

# Optional environment overrides:
# - NN_RUN_PATH
# - MODEL_RUN_PATH
ENV_RUN_PATH = os.environ.get("NN_RUN_PATH") or os.environ.get("MODEL_RUN_PATH")

LABEL_COL = "anomaly"
DROP_COLS = {LABEL_COL, "segment", "train", "sampling"}

TEST_SIZE = 0.75
BATCH_SIZE = BEST_PARAMS["batch_size"]
NB_EPOCHS = BEST_PARAMS["max_iter"]
LR = BEST_PARAMS["learning_rate_init"]
WEIGHT_DECAY = BEST_PARAMS["alpha"]
HIDDEN_LAYER_SIZES = BEST_PARAMS["hidden_layer_sizes"]
ACTIVATION_NAME = str(BEST_PARAMS["activation"]).lower()
SOLVER_NAME = str(BEST_PARAMS["solver"]).lower()
LR_POLICY = str(BEST_PARAMS["learning_rate"]).lower()
EARLY_STOPPING = BEST_PARAMS["early_stopping"]
N_ITER_NO_CHANGE = BEST_PARAMS["n_iter_no_change"]

FGM_EPS = 0.10

SAVE_MODELS = False
ARTIFACT_DIR = Path("artifacts")
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

def resolve_data_path() -> Path:
    if ENV_RUN_PATH:
        candidate = Path(ENV_RUN_PATH)
        if candidate.exists():
            return candidate
        print(f"[warn] Env path not found: {candidate}")

    if RUNS_DIR.exists():
        candidates = sorted(
            RUNS_DIR.glob(RUN_GLOB),
            key=lambda x: x.stat().st_mtime,
            reverse=True,
        )
        if candidates:
            return candidates[0]

    return DEFAULT_DATA_PATH

DATA_PATH = resolve_data_path()
print(f"Using data source: {DATA_PATH}")
print("Loaded best params:")
print(BEST_PARAMS)

Using data source: StandardizedRuns\NeuralNet_train_clean.csv
Loaded best params:
{'hidden_layer_sizes': (128, 64), 'alpha': 0.0001, 'learning_rate_init': 0.01, 'batch_size': 64, 'activation': 'relu', 'solver': 'adam', 'learning_rate': 'adaptive', 'max_iter': 1000, 'early_stopping': True, 'n_iter_no_change': 20, 'random_state': 100}


In [3]:
def load_and_prepare(csv_path: str):
    df = pd.read_csv(csv_path)

    if LABEL_COL not in df.columns:
        raise ValueError(f"Label column '{LABEL_COL}' not found in {csv_path}.")

    y = df[LABEL_COL].astype(int).to_numpy()
    feature_cols = [c for c in df.columns if c not in DROP_COLS]
    X = df[feature_cols].to_numpy(dtype=np.float32)

    if len(np.unique(y)) != 2:
        raise ValueError(f"Expected binary labels, got: {np.unique(y)}")

    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=TEST_SIZE,
        random_state=SEED,
        stratify=y,
    )

    scaler = MinMaxScaler()
    X_train = scaler.fit_transform(X_train).astype(np.float32)
    X_test = scaler.transform(X_test).astype(np.float32)

    print(f"Loaded: {csv_path}")
    print(f"Rows={len(df)}, Features={X.shape[1]}, Label dist={np.bincount(y)}")
    print(f"Train={X_train.shape}, Test={X_test.shape}")

    return X_train, X_test, y_train.astype(np.int64), y_test.astype(np.int64), feature_cols, scaler

X_train, X_test, y_train, y_test, feature_cols, scaler = load_and_prepare(str(DATA_PATH))

Loaded: StandardizedRuns\NeuralNet_train_clean.csv
Rows=1698, Features=18, Label dist=[1351  347]
Train=(424, 18), Test=(1274, 18)


In [4]:
def get_activation(name: str):
    name = name.lower()
    if name == "relu":
        return nn.ReLU
    if name == "tanh":
        return nn.Tanh
    if name == "logistic":
        return nn.Sigmoid
    raise ValueError(f"Unsupported activation for this notebook: {name}")

class MLP(nn.Module):
    def __init__(self, d_in: int, hidden_layer_sizes=HIDDEN_LAYER_SIZES, activation_name: str = ACTIVATION_NAME):
        super().__init__()

        activation_cls = get_activation(activation_name)
        layers = []
        in_features = d_in

        for hidden_units in hidden_layer_sizes:
            layers.append(nn.Linear(in_features, int(hidden_units)))
            layers.append(activation_cls())
            in_features = int(hidden_units)

        layers.append(nn.Linear(in_features, 2))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)


def make_art_classifier(
    d_in: int,
    lr: float = LR,
    weight_decay: float = WEIGHT_DECAY,
    hidden_layer_sizes=HIDDEN_LAYER_SIZES,
    activation_name: str = ACTIVATION_NAME,
):
    model = MLP(
        d_in=d_in,
        hidden_layer_sizes=hidden_layer_sizes,
        activation_name=activation_name,
    )
    criterion = nn.CrossEntropyLoss()


    return PyTorchClassifier(
        model=model,
        loss=criterion,
        optimizer=optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay),
        input_shape=(d_in,),
        nb_classes=2,
        clip_values=(0.0, 1.0),
    )


def predict_labels(art_clf: PyTorchClassifier, X: np.ndarray):
    probs = art_clf.predict(X)
    return np.argmax(probs, axis=1)


def eval_from_predictions(y_true: np.ndarray, y_pred: np.ndarray, name: str):
    acc = accuracy_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred)

    print(f"\n[{name}] acc={acc:.4f} f1={f1:.4f}")
    print("confusion matrix:")
    print(confusion_matrix(y_true, y_pred))
    print(classification_report(y_true, y_pred, digits=4))

    return {
        "model_eval": name,
        "acc": acc,
        "f1": f1,
    }


def eval_classifier(art_clf: PyTorchClassifier, X: np.ndarray, y: np.ndarray, name: str):
    y_pred = predict_labels(art_clf, X)
    metrics = eval_from_predictions(y, y_pred, name)
    return metrics, y_pred


def save_art_model_state(art_clf: PyTorchClassifier, out_path: Path):
    torch.save(art_clf.model.state_dict(), out_path)
    print(f"Saved model state: {out_path}")


In [5]:
# Step 1: clean.fit(X_train, y_train) -> trained clean model
print('Training clean model with:', {
    'hidden_layer_sizes': HIDDEN_LAYER_SIZES,
    'alpha': WEIGHT_DECAY,
    'learning_rate_init': LR,
    'batch_size': BATCH_SIZE,
    'activation': ACTIVATION_NAME,
    'solver': SOLVER_NAME,
    'learning_rate': LR_POLICY,
    'max_iter': NB_EPOCHS,
    'early_stopping': EARLY_STOPPING,
    'n_iter_no_change': N_ITER_NO_CHANGE,
    'random_state': SEED,
})

art_clean = make_art_classifier(d_in=X_train.shape[1])
art_clean.fit(X_train, y_train, batch_size=BATCH_SIZE, nb_epochs=NB_EPOCHS)

# y_pred = clean.predict(X_test)
# eval_classifier(clean, X_test, y_pred)
clean_on_clean, y_pred_clean = eval_classifier(
    art_clean,
    X_test,
    y_test,
    "clean_model_on_clean_test",
)

# save clean model
if SAVE_MODELS:
    save_art_model_state(art_clean, ARTIFACT_DIR / "nn_clean_model.pt")


Training clean model with: {'hidden_layer_sizes': (128, 64), 'alpha': 0.0001, 'learning_rate_init': 0.01, 'batch_size': 64, 'activation': 'relu', 'solver': 'adam', 'learning_rate': 'adaptive', 'max_iter': 1000, 'early_stopping': True, 'n_iter_no_change': 20, 'random_state': 100}

[clean_model_on_clean_test] acc=0.9474 f1=0.8607
confusion matrix:
[[1000   14]
 [  53  207]]
              precision    recall  f1-score   support

           0     0.9497    0.9862    0.9676      1014
           1     0.9367    0.7962    0.8607       260

    accuracy                         0.9474      1274
   macro avg     0.9432    0.8912    0.9141      1274
weighted avg     0.9470    0.9474    0.9458      1274



In [6]:
# Step 2: initialize FGM and generate adversarial samples
fgm = FastGradientMethod(estimator=art_clean, eps=FGM_EPS)

X_train_adv = fgm.generate(x=X_train)
X_test_adv = fgm.generate(x=X_test)

print("Adversarial data generated:")
print("X_train_adv:", X_train_adv.shape)
print("X_test_adv:", X_test_adv.shape)

# If y_train_adv is needed, predict it.
# The handwritten pipeline notes this as optional.
y_train_adv_pred = predict_labels(art_clean, X_train_adv)
y_test_adv_pred = predict_labels(art_clean, X_test_adv)

# For retraining and evaluation targets, keep the original ground-truth labels.
y_train_adv = y_train.copy()
y_test_adv = y_test.copy()

# Build combined clean + adversarial datasets.
X_train_combined = np.concatenate([X_train, X_train_adv], axis=0).astype(np.float32)
y_train_combined = np.concatenate([y_train, y_train_adv], axis=0).astype(np.int64)

X_test_combined = np.concatenate([X_test, X_test_adv], axis=0).astype(np.float32)
y_test_combined = np.concatenate([y_test, y_test_adv], axis=0).astype(np.int64)

print("Combined datasets created:")
print("X_train_combined:", X_train_combined.shape)
print("y_train_combined:", y_train_combined.shape)
print("X_test_combined:", X_test_combined.shape)
print("y_test_combined:", y_test_combined.shape)


Adversarial data generated:
X_train_adv: (424, 18)
X_test_adv: (1274, 18)
Combined datasets created:
X_train_combined: (848, 18)
y_train_combined: (848,)
X_test_combined: (2548, 18)
y_test_combined: (2548,)


In [7]:
# Evaluate performance of the clean model on adversarial and combined test data
clean_on_adv, y_pred_adv_clean_model = eval_classifier(
    art_clean,
    X_test_adv,
    y_test_adv,
    "clean_model_on_adv_test",
)

clean_on_combined, y_pred_combined_clean_model = eval_classifier(
    art_clean,
    X_test_combined,
    y_test_combined,
    "clean_model_on_combined_test",
)



[clean_model_on_adv_test] acc=0.1397 f1=0.1672
confusion matrix:
[[ 68 946]
 [150 110]]
              precision    recall  f1-score   support

           0     0.3119    0.0671    0.1104      1014
           1     0.1042    0.4231    0.1672       260

    accuracy                         0.1397      1274
   macro avg     0.2080    0.2451    0.1388      1274
weighted avg     0.2695    0.1397    0.1220      1274


[clean_model_on_combined_test] acc=0.5436 f1=0.3528
confusion matrix:
[[1068  960]
 [ 203  317]]
              precision    recall  f1-score   support

           0     0.8403    0.5266    0.6475      2028
           1     0.2482    0.6096    0.3528       520

    accuracy                         0.5436      2548
   macro avg     0.5443    0.5681    0.5001      2548
weighted avg     0.7195    0.5436    0.5873      2548



In [8]:
# Retrain using ART's AdversarialTrainer with the FGM attack
art_adv = make_art_classifier(d_in=X_train.shape[1])

adv_trainer = AdversarialTrainer(
    classifier=art_adv,
    attacks=fgm,
    ratio=.55,  # Use 50% adversarial samples during training
)

adv_trainer.fit(
    X_train,
    y_train,
    batch_size=BATCH_SIZE,
    nb_epochs=NB_EPOCHS,
)

if SAVE_MODELS:
    save_art_model_state(art_adv, ARTIFACT_DIR / "nn_adversarial_trained_model.pt")


Adversarial training epochs: 100%|██████████| 1000/1000 [00:20<00:00, 49.26it/s]


In [9]:
# Test using X_test_adv and the combined clean+adv test set
adv_trained_on_adv, y_pred_adv = eval_classifier(
    art_adv,
    X_test_adv,
    y_test_adv,
    "adv_trained_model_on_adv_test",
)

# Also check whether adversarial training preserved clean performance
adv_trained_on_clean, y_pred_clean_adv_model = eval_classifier(
    art_adv,
    X_test,
    y_test,
    "adv_trained_model_on_clean_test",
)

adv_trained_on_combined, y_pred_combined_adv_model = eval_classifier(
    art_adv,
    X_test_combined,
    y_test_combined,
    "adv_trained_model_on_combined_test",
)



[adv_trained_model_on_adv_test] acc=0.9396 f1=0.8344
confusion matrix:
[[1003   11]
 [  66  194]]
              precision    recall  f1-score   support

           0     0.9383    0.9892    0.9630      1014
           1     0.9463    0.7462    0.8344       260

    accuracy                         0.9396      1274
   macro avg     0.9423    0.8677    0.8987      1274
weighted avg     0.9399    0.9396    0.9368      1274


[adv_trained_model_on_clean_test] acc=0.9466 f1=0.8559
confusion matrix:
[[1004   10]
 [  58  202]]
              precision    recall  f1-score   support

           0     0.9454    0.9901    0.9672      1014
           1     0.9528    0.7769    0.8559       260

    accuracy                         0.9466      1274
   macro avg     0.9491    0.8835    0.9116      1274
weighted avg     0.9469    0.9466    0.9445      1274


[adv_trained_model_on_combined_test] acc=0.9431 f1=0.8453
confusion matrix:
[[2007   21]
 [ 124  396]]
              precision    recall  f1-scor

In [10]:
# Summary table
summary_df = pd.DataFrame([
    clean_on_clean,
    clean_on_adv,
    clean_on_combined,
    adv_trained_on_clean,
    adv_trained_on_adv,
    adv_trained_on_combined,
])

summary_df


,model_eval,acc,f1
0,clean_model_on_clean_test,0.947410,0.860707
1,clean_model_on_adv_test,0.139717,0.167173
2,clean_model_on_combined_test,0.543564,0.352810
3,adv_trained_model_on_clean_test,0.946625,0.855932
4,adv_trained_model_on_adv_test,0.939560,0.834409
5,adv_trained_model_on_combined_test,0.943093,0.845251


In [11]:
# Save metrics
out_csv = Path(r"Results\NeuralNetworksResults\nn_fgm_evasion_pipeline_.csv")
out_csv.parent.mkdir(parents=True, exist_ok=True)
summary_df.to_csv(out_csv, index=False)
print(f"Saved: {out_csv}")


Saved: Results\NeuralNetworksResults\nn_fgm_evasion_pipeline_.csv


## Consistency Gate

This implements the simple ADEPTRS consistency gate:

- **Anomaly** if both models flag anomaly
- **AttackFlag** if only the robust model flags anomaly
- **Nominal** otherwise

The clean model is the **Nominal Specialist** and the adversarially trained model is the **Robust Guardian**.

In [12]:
# Simple consistency gate
TA = 0.90   # nominal specialist anomaly threshold
TB = 0.60   # robust guardian anomaly threshold

def get_anomaly_probs(art_clf: PyTorchClassifier, X: np.ndarray):
    probs = art_clf.predict(X)
    return probs[:, 1]

def consistency_gate(art_nominal: PyTorchClassifier, art_robust: PyTorchClassifier, X: np.ndarray, ta: float = TA, tb: float = TB):
    p_nominal_anom = get_anomaly_probs(art_nominal, X)
    p_robust_anom = get_anomaly_probs(art_robust, X)

    nominal_flags = (p_nominal_anom >= ta).astype(int)
    robust_flags = (p_robust_anom >= tb).astype(int)

    gate_labels = []
    for m1, m2 in zip(nominal_flags, robust_flags):
        if m1 == 1 and m2 == 1:
            gate_labels.append("Anomaly")
        elif m1 == 0 and m2 == 1:
            gate_labels.append("AttackFlag")
        else:
            gate_labels.append("Nominal")

    gate_df = pd.DataFrame({
        "p_nominal_anom": p_nominal_anom,
        "p_robust_anom": p_robust_anom,
        "nominal_flag": nominal_flags,
        "robust_flag": robust_flags,
        "gate_label": gate_labels,
    })

    return gate_df

def summarize_gate(gate_df: pd.DataFrame, y_true: np.ndarray, name: str):
    # For anomaly-detection metrics, only "Anomaly" counts as predicted anomaly.
    y_gate_binary = (gate_df["gate_label"] == "Anomaly").astype(int).to_numpy()

    acc = accuracy_score(y_true, y_gate_binary)
    f1 = f1_score(y_true, y_gate_binary, zero_division=0)
    attack_flag_rate = (gate_df["gate_label"] == "AttackFlag").mean()
    anomaly_rate = (gate_df["gate_label"] == "Anomaly").mean()
    nominal_rate = (gate_df["gate_label"] == "Nominal").mean()

    print(f"\n[{name}] acc={acc:.4f} f1={f1:.4f}")
    print("gate label counts:")
    print(gate_df["gate_label"].value_counts())

    return {
        "gate_eval": name,
        "acc": acc,
        "f1": f1,
        "attack_flag_rate": attack_flag_rate,
        "anomaly_rate": anomaly_rate,
        "nominal_rate": nominal_rate,
    }

gate_clean_df = consistency_gate(art_clean, art_adv, X_test)
gate_adv_df = consistency_gate(art_clean, art_adv, X_test_adv)
gate_combined_df = consistency_gate(art_clean, art_adv, X_test_combined)

gate_clean_summary = summarize_gate(gate_clean_df, y_test, "gate_on_clean_test")
gate_adv_summary = summarize_gate(gate_adv_df, y_test_adv, "gate_on_adv_test")
gate_combined_summary = summarize_gate(gate_combined_df, y_test_combined, "gate_on_combined_test")

gate_summary_df = pd.DataFrame([
    gate_clean_summary,
    gate_adv_summary,
    gate_combined_summary,
])

gate_summary_df


[gate_on_clean_test] acc=0.9317 f1=0.8009
gate label counts:
gate_label
Nominal       1077
Anomaly        177
AttackFlag      20
Name: count, dtype: int64

[gate_on_adv_test] acc=0.8367 f1=0.3500
gate label counts:
gate_label
Nominal       1073
AttackFlag     141
Anomaly         60
Name: count, dtype: int64

[gate_on_combined_test] acc=0.8842 f1=0.6103
gate label counts:
gate_label
Nominal       2150
Anomaly        237
AttackFlag     161
Name: count, dtype: int64


,gate_eval,acc,f1,attack_flag_rate,anomaly_rate,nominal_rate
0,gate_on_clean_test,0.931711,0.800915,0.015699,0.138932,0.845369
1,gate_on_adv_test,0.836735,0.350000,0.110675,0.047096,0.842229
2,gate_on_combined_test,0.884223,0.610304,0.063187,0.093014,0.843799


In [13]:
# Save gate outputs
gate_out_csv = Path(r"Results\NeuralNetworksResults\nn_consistency_gate_summary.csv")
gate_out_csv.parent.mkdir(parents=True, exist_ok=True)
gate_summary_df.to_csv(gate_out_csv, index=False)

gate_clean_df.to_csv(gate_out_csv.parent / "nn_gate_clean_predictions.csv", index=False)
gate_adv_df.to_csv(gate_out_csv.parent / "nn_gate_adv_predictions.csv", index=False)
gate_combined_df.to_csv(gate_out_csv.parent / "nn_gate_combined_predictions.csv", index=False)

print(f"Saved: {gate_out_csv}")

Saved: Results\NeuralNetworksResults\nn_consistency_gate_summary.csv
